In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.graph_inference import Graph as Graph_interference
from src.models.graph_training import Graph as Graph_train

# from src.data_loader.transforms_numpy import polar2cartesian, addSpeckleNoise, energyLoss, addBandReflects




- Create instances of graphs
- Load config files

In [3]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

with open(model_config_pth, "r") as f:
    model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
    sonar_config = Box(yaml.safe_load(f))

with open(train_config_pth, "r") as f:
    train_config = Box(yaml.safe_load(f))


PatchGraph_i = Graph_interference(model_config, sonar_config)
PatchGraph_t = Graph_train(model_config, sonar_config, train_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

- prepare data 

In [4]:
# test data 

data_pth_sim = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls/200.png'

# read frame 
frame_sim = cv2.imread(data_pth_sim, 0)

frame_sim = frame_sim.astype(np.uint8)
frame_sim = torch.from_numpy(frame_sim) # convert to torch tensor
frame_sim = frame_sim.float().unsqueeze(0).unsqueeze(0).unsqueeze(0)


frames_inm_series = 5
batch_size = 1
t_0 = 1.0
dt = 0.5

frame_sim_b = torch.cat([frame_sim for _ in range(frames_inm_series)], axis = 1)
frame_sim_b = torch.cat([frame_sim_b for _ in range(batch_size)], axis = 0)


time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_inm_series)], device = device, dtype = torch.float)
time_tensor = time_tensor.unsqueeze(0)
time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)


print('-'*80)
print(f'Input data format:')
print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
print(f'time tensor: {time_tensor.shape}')
print('-'*80)

--------------------------------------------------------------------------------
Input data format:
simulated tensor shape: torch.Size([1, 1, 1, 800, 768]), data type: torch.float32
simulated tensors batch shape: torch.Size([1, 5, 1, 800, 768]), data type: torch.float32
time tensor: torch.Size([1, 5])
--------------------------------------------------------------------------------


## Output test


In [5]:
def compare_shuffled(t1, t2):
    if t1.shape != t2.shape: return False
    # Spłaszczamy do 2D (N, -1) i sortujemy wiersze leksykograficznie
    t1_f = t1.reshape(t1.size(0), -1)
    t2_f = t2.reshape(t2.size(0), -1)
    for i in range(t1_f.shape[1]-1, -1, -1):
        t1_f = t1_f[t1_f[:, i].argsort(stable=True)]
        t2_f = t2_f[t2_f[:, i].argsort(stable=True)]
    return torch.allclose(t1_f.float(), t2_f.float(), atol=1e-3)

In [6]:
# === traning graph === 
t_start = time.time()

poses, coords_phi = PatchGraph_t.append(frame_sim_b, time_tensor, device)
corr_t, ctx_t, source_frame_idx_t, target_frame_idx_t, patch_idx_t = PatchGraph_t.update_step(poses, coords_phi,  device)

traning_time = time.time() - t_start

In [7]:
# === inference graph === 
t_start = time.time()

b, n, c, h, w = frame_sim_b.shape
print(frame_sim_b.shape)
for batch in range(b):
    for frame_num in range(n):
        frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
        t = time_tensor[batch, frame_num]

        PatchGraph_i.append(frame, t, device)
        corr, ctx, source_frame_idx, target_frame_idx, patch_idx = PatchGraph_i.update_step(device)

corr_i, ctx_i, source_frame_idx_i, target_frame_idx_i, patch_idx_i = corr, ctx, source_frame_idx, target_frame_idx, patch_idx
inference_time = time.time() - t_start

    

torch.Size([1, 5, 1, 800, 768])


In [ ]:
# comparision:
print('='*80)
print(f'Time: traning: {traning_time}, inference: {inference_time}')



print(f'number if edges in traning: {corr_t.shape}')
print(f'number if edges in inference: {corr_i.shape}')

output_test = []

# output_test.append('Passed') if compare_shuffled(corr_t, corr_i) else output_test.append('Failed')
# print(f'corr test: {output_test[0]}')

# output_test.append('Passed') if compare_shuffled(ctx_t, ctx_i) else output_test.append('Failed')
# print(f'ctx test: {output_test[1]}')

output_test.append('Passed') if compare_shuffled(source_frame_idx_t, source_frame_idx_i) else output_test.append('Failed')
print(f'source_frame_idx test: {output_test[2]}')

output_test.append('Passed') if compare_shuffled(target_frame_idx_t, target_frame_idx_i) else output_test.append('Failed')
print(f'source_frame_idx test: {output_test[3]}')

output_test.append('Passed') if compare_shuffled(patch_idx_t, patch_idx_i) else output_test.append('Failed')
print(f'source_frame_idx test: {output_test[4]}')

Time: traning: 2.879422903060913, inference: 6.278807163238525
number if edges in traning: torch.Size([90, 50])
number if edges in inference: torch.Size([90, 50])
corr test: Failed
ctx test: Failed
corr test: Passed
corr test: Passed
corr test: Passed


In [9]:


# poses, coords_phi = PatchGraph_t.append(frame_sim_b, time_tensor, device)

# corr, ctx, source_frame_idx, target_frame_idx, patch_idx = PatchGraph_t.update_step(poses, coords_phi,  device)

# print('='*40)
# print(f'    corr: {corr.shape}')
# print(f'    ctx: {ctx.shape}')
# print(f'    source frame: {source_frame_idx.shape}')
# print(f'    target frame: {target_frame_idx.shape}')
# print(f'    patch idx: {patch_idx.shape}')

# # print('='*40)

In [10]:
# # ----
# PatchGraph_i = Graph_interference(model_config, sonar_config)
# single_batch = frame_sim_b[0, ...]
# single_time = time_tensor[0, ...]

# print(single_time)
# frames_num = single_time.shape[0]
# for _ in range(2):
#     for i in range(frames_num):
#         t_start = time.time()
#         frame = single_batch[i, ...].unsqueeze(0).unsqueeze(0)
#         ti = single_time[i].unsqueeze(0).unsqueeze(0)
    
#         PatchGraph_i.append(frame, ti, device)
    
#         corr, ctx, source_frame_idx, target_frame_idx, patch_idx = PatchGraph_i.update_step(device)

#         print(f'exec time: {time.time() - t_start}')
    